# Lab 9.4 &mdash; Spans You Can Bill

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Price a call and put the number on the span
- Work out which questions your instrumentation can answer, and which it cannot
- Count the time series a label adds before you add it
- Do the arithmetic that shows head sampling loses the incident

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, and none of them needs a cluster, so your
> score never depends on a live endpoint or on `kubectl` working. Cells marked **Run it for real**
> do call the sandbox model or your namespace; if either is unreachable they print how to fix it
> instead of crashing.

> **Instrumentation is a decision made before the incident.** Every question in this
> lab is answerable or not depending on an attribute somebody chose to record weeks
> earlier, when nothing was wrong.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-9-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- graded cells still work)")
print("namespace:", APP_NS or "(unknown -- graded cells still work)")

## Concept

OpenTelemetry gives three signals and one vocabulary.

- A **trace** is one request, as a tree of **spans**. It answers *where did the time go, on this
  one*.
- A **metric** is a number over a window, cut by **labels**. It answers *how often, how bad,
  across all of them*.
- A **log** is an event with a timestamp. It answers *what exactly happened at 14:07*.

They compose: the trace ID goes in the log line, the span carries the attributes, the metric is
derived from the spans. An agent adds a fourth thing that none of the three pillars gives you
for free &mdash; **what it decided and what that cost** &mdash; and that is what this lab is about.

## Section 1 &mdash; The number that has to be on the span

Cost is a per-request property. It cannot be recovered later from a monthly invoice, and it
cannot be divided by request count &mdash; the whole point is that requests differ.

In [ ]:
# USD per 1,000 tokens. Illustrative rates; the shape is what matters.
RATES = {
    "qwen-lab":  {"in": 0.0002, "out": 0.0006},
    "big-model": {"in": 0.0030, "out": 0.0150},
}


def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    """What one model call cost."""
    r = RATES[model]
    # TODO: input and output tokens are priced differently, and the rates above are per
    # 1,000 tokens. Return the cost of this one call.
    return BLANK


def model_span(name: str, model: str, input_tokens: int, output_tokens: int,
               duration_s: float, **extra) -> dict:
    """One span in OpenTelemetry's shape, with the attributes an agent needs."""
    return {
        "name": name,
        "duration_s": duration_s,
        "attrs": {
            "gen_ai.request.model": model,
            "gen_ai.usage.input_tokens": input_tokens,
            "gen_ai.usage.output_tokens": output_tokens,
            "app.cost_usd": cost_usd(model, input_tokens, output_tokens),
            **extra,
        },
    }

In [ ]:
# --- Self-check: Section 1
check("a thousand tokens each way on the lab model costs 0.0008",
      lambda: round(cost_usd("qwen-lab", 1000, 1000), 6) == 0.0008)
check("output tokens cost three times input on that model",
      lambda: round(cost_usd("qwen-lab", 0, 1000), 10)
              == round(3 * cost_usd("qwen-lab", 1000, 0), 10))
check("the same call on the big model costs 0.018",
      lambda: round(cost_usd("big-model", 1000, 1000), 6) == 0.018)
check("...which is 22.5x, and that ratio is a routing decision",
      lambda: round(cost_usd("big-model", 1000, 1000) / cost_usd("qwen-lab", 1000, 1000), 1)
              == 22.5)
check("a call with no tokens costs nothing",
      lambda: cost_usd("qwen-lab", 0, 0) == 0)
check("the span carries the cost as an attribute, not as a note in a log line",
      lambda: "app.cost_usd" in model_span("plan", "qwen-lab", 900, 120, 2.1)["attrs"])
check("...and the token counts too, so the cost can be re-derived when rates change",
      lambda: model_span("plan", "qwen-lab", 900, 120, 2.1)["attrs"]
              ["gen_ai.usage.input_tokens"] == 900,
      "prices change; recording only the dollar figure makes the history unusable")

## Section 2 &mdash; What your instrumentation can answer

Here is one hour of a deployed service. Every request is a trace; the spans carry what somebody
decided to record.

In [ ]:
import random

def build_window(n: int = 240, seed: int = 9) -> list:
    """One hour of traffic. Deterministic, so everyone's numbers match."""
    rng = random.Random(seed)
    tenants = ["ops-emea", "ops-apac", "ops-us", "treasury", "client-desk"]
    out = []
    for i in range(n):
        model = "big-model" if rng.random() < 0.15 else "qwen-lab"
        pt, ct = rng.randint(600, 2400), rng.randint(80, 700)
        ok = rng.random() > 0.005
        out.append({
            "trace_id": f"{i:08x}",
            "tenant": rng.choice(tenants),
            "endpoint": rng.choice(["/ask", "/ask", "/ask", "/investigate"]),
            "payment_ref": f"PMT-{rng.randint(1000, 2200)}",
            "model": model,
            "input_tokens": pt,
            "output_tokens": ct,
            "cost_usd": cost_usd(model, pt, ct),
            "duration_s": round(rng.uniform(1.5, 12.0), 2),
            "ok": ok,
        })
    return out


# Built lazily: cost_usd is blanked above, so building at import time would crash the cell
# rather than report a [TODO].
_window = []
def window() -> list:
    if not _window:
        _window.extend(build_window())
    return _window


def spend_by(field: str) -> dict:
    """Total cost grouped by one recorded field."""
    out = {}
    for r in window():
        out[r[field]] = round(out.get(r[field], 0.0) + r["cost_usd"], 4)
    return out

In [ ]:
# What this instrumentation recorded -- and therefore what it can be asked.
RECORDED = {"tenant", "endpoint", "model", "input_tokens", "output_tokens",
            "cost_usd", "duration_s", "ok", "trace_id", "payment_ref"}

QUESTIONS = {
    "what did treasury spend this hour?":        {"tenant", "cost_usd"},
    "which model is the money going to?":        {"model", "cost_usd"},
    "how slow is the 95th percentile?":          {"duration_s"},
    "how often did a guardrail refuse?":         {"decision"},
    "did the answer cite a policy document?":    {"cited_policy"},
    "was the answer any good?":                  {"score"},
}

def can_answer(question: str) -> bool:
    """A question is answerable only if every attribute it needs was recorded."""
    return QUESTIONS[question] <= RECORDED

In [ ]:
# --- Self-check: Section 2
check("cost per tenant is answerable",
      lambda: can_answer("what did treasury spend this hour?"))
check("...and treasury is not the biggest spender",
      lambda: max(spend_by("tenant"), key=spend_by("tenant").get) != "treasury")
check("the model split is answerable",
      lambda: can_answer("which model is the money going to?"))
check("AN EIGHTH OF THE CALLS ARE THREE QUARTERS OF THE BILL",
      lambda: spend_by("model")["big-model"] > 2 * spend_by("model")["qwen-lab"],
      "a routing decision worth finding, and only visible because the model is on the span")
check("latency percentiles are answerable",
      lambda: can_answer("how slow is the 95th percentile?"))
check("but the refusal rate is NOT",
      lambda: not can_answer("how often did a guardrail refuse?"),
      "Module 8's control is invisible here -- nobody recorded the decision")
check("nor whether the answer cited anything",
      lambda: not can_answer("did the answer cite a policy document?"),
      "Module 6's grounding check, missing from Module 9's telemetry")
check("nor whether it was any good",
      lambda: not can_answer("was the answer any good?"))
check("three of six questions cannot be answered at any price",
      lambda: sum(1 for q in QUESTIONS if not can_answer(q)) == 3,
      "not slowly, not expensively -- the data does not exist")

def _pillars():
    s = spend_by("model")
    share = s["big-model"] / sum(s.values())
    n_big = sum(1 for r in window() if r["model"] == "big-model")
    print(f"  spend by model : {s}")
    print(f"  big-model      : {n_big}/{len(window())} calls, {share:.0%} of the spend")
    print(f"  spend by tenant: {spend_by('tenant')}")
    print(f"  total this hour: ${sum(s.values()):.2f}   "
          f"-> ${sum(s.values()) * 24 * 30:,.0f}/month at this rate")
    print()
    for q in QUESTIONS:
        print(f"  {'yes' if can_answer(q) else 'NO ':4} {q}")
guard(_pillars)

### Read it

The three unanswerable questions are the agent-specific ones. Latency, cost and error rate come
free with any HTTP instrumentation; *did a guardrail fire*, *did the answer cite its source* and
*was it right* have to be recorded on purpose, by you, as span attributes.

That is the whole of the difference between observability for a web service and AgentOps. The
system can be perfectly healthy on all three pillars and be answering wrongly &mdash; and Module
8 closed with exactly that slide.

## Section 3 &mdash; Labels multiply, so count before you add one

`payment_ref` is on the span and that is correct. Putting it on a **metric** is a different act
with a different cost, because every distinct value creates a time series that is stored,
indexed and queried forever.

In [ ]:
CARDINALITY = {"service": 1, "endpoint": 4, "status": 3, "model": 2,
               "tenant": 5, "payment_ref": 1200}

def series_count(labels) -> int:
    """How many time series does one metric with these labels produce?"""
    # TODO: labels do not add. Each one multiplies the number of series by the number
    # of distinct values it can take.
    return BLANK


def safe_to_label(labels, budget: int = 500) -> bool:
    """Would this label set stay inside the series budget for one metric?"""
    return series_count(labels) <= budget

In [ ]:
# --- Self-check: Section 3
BASE = ["service", "endpoint", "status"]

check("the base label set is twelve series",
      lambda: series_count(BASE) == 12)
check("adding the model doubles it",
      lambda: series_count(BASE + ["model"]) == 24)
check("adding the tenant is still fine",
      lambda: series_count(BASE + ["model", "tenant"]) == 120)
check("ADDING THE PAYMENT REFERENCE IS 144,000 SERIES",
      lambda: series_count(BASE + ["model", "tenant", "payment_ref"]) == 144000,
      "for one metric -- and payment_ref is unbounded, so that number only grows")
check("the budget check catches it",
      lambda: safe_to_label(BASE + ["model", "tenant"])
              and not safe_to_label(BASE + ["payment_ref"]))
check("the same field on a SPAN costs nothing extra",
      lambda: "payment_ref" in RECORDED,
      "spans are stored per request; labels are stored per distinct combination, forever")

def _labels():
    for extra in ([], ["model"], ["model", "tenant"], ["model", "tenant", "payment_ref"]):
        ls = BASE + extra
        print(f"  {series_count(ls):>7,} series  {'ok ' if safe_to_label(ls) else 'NO '}"
              f" {'+'.join(ls)}")
guard(_labels)

## Section 4 &mdash; The trace you need is the one you did not keep

Traces are the expensive signal, so everybody samples. Head sampling &mdash; decide at the start
of the request, keep 10% &mdash; is the default because it is the cheapest to implement.

Do the arithmetic on it once and you will not use it for an agent service.

In [ ]:
DAILY_REQUESTS = 2000
FAILURE_RATE   = 1 / 200          # the thing you will be asked about

def expected_captured(p: float, requests: int = DAILY_REQUESTS,
                      failure_rate: float = FAILURE_RATE) -> float:
    """How many of the day's failures head sampling at rate p keeps."""
    return requests * failure_rate * p


def p_miss_everything(p: float, requests: int = DAILY_REQUESTS,
                      failure_rate: float = FAILURE_RATE) -> float:
    """The chance that a whole day of head sampling keeps NOT ONE failing trace."""
    failures = requests * failure_rate
    # TODO: each failure is kept independently with probability p. What is the chance
    # that every single one of them is dropped?
    return BLANK


def tail_kept_fraction(p_normal: float, failure_rate: float = FAILURE_RATE) -> float:
    """Tail sampling: decide when the request ENDS. Keep every failure, and p of the rest."""
    return failure_rate + p_normal * (1 - failure_rate)

In [ ]:
# --- Self-check: Section 4
check("there are ten failures in a day at this rate",
      lambda: DAILY_REQUESTS * FAILURE_RATE == 10)
check("head sampling at 10% expects to keep exactly one of them",
      lambda: expected_captured(0.10) == 1.0)
check("...and on 35% of days it keeps none at all",
      lambda: round(p_miss_everything(0.10), 4) == 0.3487,
      "one day in three, the trace the incident review asks for was never stored")
check("keeping everything misses nothing, and costs everything",
      lambda: p_miss_everything(1.0) == 0.0)
check("even 50% head sampling loses every failure on 1 day in 1000",
      lambda: round(p_miss_everything(0.50), 4) == 0.001)
check("TAIL SAMPLING AT 5% KEEPS EVERY FAILURE",
      lambda: tail_kept_fraction(0.05) > FAILURE_RATE,
      "the decision moves to the end of the request, when the outcome is known")
check("...while storing less than head sampling at 10%",
      lambda: tail_kept_fraction(0.05) < 0.10)
check("...about 45% less",
      lambda: round(1 - tail_kept_fraction(0.05) / 0.10, 2) == 0.45)

def _sampling():
    print(f"  {'strategy':28} {'stored':>8} {'failures kept':>14} {'blind days':>11}")
    for p in (0.01, 0.10, 0.50):
        print(f"  head sampling at {p:>4.0%}         {p:>7.1%} "
              f"{expected_captured(p):>13.1f} {p_miss_everything(p):>10.1%}")
    for p in (0.01, 0.05):
        f = tail_kept_fraction(p)
        print(f"  tail sampling, {p:>3.0%} of clean {f:>7.1%} "
              f"{DAILY_REQUESTS * FAILURE_RATE:>13.1f} {0.0:>10.1%}")
guard(_sampling)

### Read it

Head sampling at 10% is the industry default and it loses the entire day's evidence about one
day in three. That is not a tail risk, it is a coin you flip every incident review.

Tail sampling costs more to run &mdash; the collector must buffer each trace until the request
finishes, which is why this decision belongs in the **collector** and not in your application.
That is the practical reason the OTLP collector exists between your process and your backend: it
is the one place where sampling, redaction and fan-out to several destinations can happen
without a redeploy of the service.

For an agent, extend the keep rule past errors: keep every trace that **refused**, every one that
**escalated to a human**, and every one in the slowest 1%. Those are the three that get asked
about, and none of them is an error.

## Run it for real

Send one trace to LangFuse, with the cost attributes on it. Your sandbox is pointed at a shared
project and separated by environment, so you will see your own traces and not anybody else's.

In [ ]:
def send_trace():
    host = os.environ.get("LANGFUSE_HOST")
    if not (host and os.environ.get("LANGFUSE_PUBLIC_KEY")
            and os.environ.get("LANGFUSE_SECRET_KEY")):
        print("LangFuse is not configured here. To point at one, set LANGFUSE_HOST,")
        print("LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY. Nothing above needed it.")
        return
    from langfuse import Langfuse
    lf = Langfuse()                     # reads LANGFUSE_HOST / _PUBLIC_KEY / _SECRET_KEY
    if not lf.auth_check():
        print("LangFuse credentials rejected.")
        return
    spans = [model_span("plan",     "qwen-lab",  900, 120, 2.1, step="plan"),
             model_span("retrieve", "qwen-lab",  240,  40, 0.4, step="retrieve"),
             model_span("answer",   "big-model", 2100, 480, 6.8, step="answer")]
    # SDK 4.x: observations nest by being entered inside one another. client.trace(...)
    # is the v3 API and does not exist here.
    with lf.start_as_current_observation(name="investigate-payment", as_type="span") as root:
        root.update(metadata={"app.cost_usd": round(sum(s["attrs"]["app.cost_usd"]
                                                        for s in spans), 6),
                              "tenant": "ops-emea", "payment_ref": "PMT-1003"})
        for s in spans:
            with lf.start_as_current_observation(name=s["name"], as_type="span") as obs:
                obs.update(metadata=s["attrs"])
    lf.flush()
    env = os.environ.get("LANGFUSE_TRACING_ENVIRONMENT", "(unset)")
    print(f"sent one trace to {host}")
    print(f"environment tag: {env}  -- filter on it in the UI to see only your own")
    print("Open the trace and check the cost is on the root as well as the leaves.")

guard(send_trace)

### Read it

Note what separated your traces from everyone else's: an environment variable, read by the SDK,
with no code of yours involved. That is worth copying &mdash; per-tenant or per-environment
separation that depends on every call site remembering to pass a field is separation that lasts
until the first new call site.

The same trace, sent through an OTLP collector to a vendor backend, would carry the same
attributes under the same `gen_ai.*` names. That naming convention is why the choice of backend
is reversible and the choice of *what to record* is not.

In [ ]:
score()

## Your turn

1. Add `decision` and `cited_policy` to `RECORDED` and to `model_span`, then re-run Section 2.
   Three questions become answerable; work out what it would have cost to add them after the
   incident instead of before.
2. Cost is on the span. Write the aggregation that turns it into a per-tenant daily figure, and
   decide which of the two &mdash; span attribute or metric label &mdash; `tenant` should be.
   (You may want both, for different reasons.)
3. Design the keep rule for tail sampling in your own words: errors, refusals, escalations,
   slowest 1%. Then estimate the stored fraction using `tail_kept_fraction` with a failure rate
   that includes all four.